# Stiffness 3 — steep fronts

A **step** into a column of pure transport. One front, and nothing else in the model: no binding, no particle, no second section. The sharpness of that front is set by the axial dispersion alone.

| | |
|---|---|
| bed length / diameter | 0.1 m / 0.01 m |
| superficial velocity | 4.2·10⁻⁴ m/s (150 cm/h) |
| bed porosity | 0.37 |
| feed | 1 mol/m³ from $t = 0$, never switched off |
| discretization | finite volumes with WENO — monotone, so no oscillations to blame |

Dispersion broadens a front as it travels, so after crossing the column its width is about

$$w \;\approx\; \sqrt{2 D \, L/u},$$

and the grid can only represent what is wider than one cell. Those are the two knobs: **how sharp the front is**, and **whether the grid can see it**.

In [ ]:
import utility.steep_front_explorer as explorer

explorer.show_setting()

## Two indicators, pulling in opposite directions

This notebook reports the classical **stiffness ratio** — fastest process over slowest — next to the **front width in cells**. They disagree, and that disagreement is the lesson.

Cutting the dispersion makes the stiffness ratio rise steeply, because the slowest process in this model *is* the dispersion. By the textbook definition the system becomes dramatically stiffer. Meanwhile the front narrows, and at some point it slips below the size of a cell.

Only one of the two predicts what the simulation costs. Sweep both knobs and watch which.

In [ ]:
from ipywidgets import interact

show_front = explorer.make_explorer()

interact(show_front, **show_front.sliders);

## What happened

**This is the first knob in the chapter that costs real money.** On 160 cells, taking the dispersion from 10⁻⁵ to 10⁻¹⁰ m²/s takes the step count from 615 to 3965 — a factor of **6.5**. On 640 cells it goes from 697 to 10005, a factor of **14.4**.

**But only while the grid can see the front.** The cost stops climbing the moment the front drops below about one cell, and it does so on every grid:

| cells | cost | stops climbing at |
|---|---|---|
| 40 | ×2.9 | front width 0.53 cells |
| 160 | ×6.5 | front width 0.67 cells |
| 640 | ×14.4 | front width 0.85 cells |

Past that point the dispersion can be cut another two decades for free. The solver is not resolving a sharper front any more — it is resolving the sharpest front *this grid can represent*, and that has stopped changing.

**And the stiffness ratio was lying the whole time.** It rises by five decades across the sweep while the cost saturates. It rises because the slowest process is getting slower, and a slow process costs nothing — exactly as the rates section said. The quantity that actually predicts the bill is the front width in cells.

**The BDF order never drops**, staying between 4.96 and 5.00 throughout. As everywhere else in this chapter, these are steps chosen for accuracy, not forced by stability.

### The same thing happens when the chemistry makes the front

Here the front is sharp because the feed was a step and the dispersion is low. A nonlinear isotherm can produce one on its own: in an overloaded Langmuir system the back of a band travels faster than its front, so the front **self-sharpens** as it moves, no matter how smooth the injection was. Measured on the two component Langmuir benchmark, cutting the dispersion from 10⁻⁷ to 10⁻¹¹ costs ×1.27 on a coarse grid and ×2.05 on a fine one — the same mechanism, the same dependence on resolution, with the isotherm playing the part of the step input.

That case is harder to read because the loading, the isotherm and the grid all move the front at once, which is why this notebook isolates it with a step instead.

> **Take-away.** A steep front is the one thing in this chapter that reliably costs you, and what it costs depends on your grid: fine enough to resolve it, and you pay; too coarse, and you get the bill waived along with the accuracy. Before blaming the time integrator for a slow run, ask how sharp your solution is and how many cells are sitting across the sharpest part of it.